# Fase 2 - Preparacion NIH ChestX-ray14 en Kaggle

Notebook base para preparar metadata multilabel y splits por paciente. No descarga el dataset, no entrena modelos y no guarda imagenes en el repositorio.

## 1. Verificar entorno

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd

print('Python:', sys.version)
print('CWD:', Path.cwd())
print('/kaggle/input exists:', Path('/kaggle/input').exists())

## 2. Clonar o usar repo

In [ ]:
# Si este notebook no esta dentro del repo, descomenta y ajusta la URL:
# !git clone https://github.com/USUARIO/Tesis_CXR.git /kaggle/working/Tesis_CXR

candidate_repos = [Path.cwd(), Path('/kaggle/working/Tesis_CXR')]
REPO_DIR = next((path for path in candidate_repos if (path / 'scripts').exists()), None)

if REPO_DIR is None:
    raise FileNotFoundError('No se encontro el repositorio. Clona Tesis_CXR en /kaggle/working/Tesis_CXR.')

print('Repo:', REPO_DIR)

## 3. Detectar ruta del dataset en /kaggle/input

In [ ]:
input_root = Path('/kaggle/input')
metadata_candidates = sorted(input_root.rglob('Data_Entry_2017.csv')) if input_root.exists() else []

if not metadata_candidates:
    raise FileNotFoundError('Agrega NIH ChestX-ray14 como Input de Kaggle y verifica Data_Entry_2017.csv.')

METADATA_CSV = metadata_candidates[0]
IMAGES_ROOT = METADATA_CSV.parent
OUTPUT_DIR = Path('/kaggle/working/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Metadata CSV:', METADATA_CSV)
print('Images root:', IMAGES_ROOT)
print('Output dir:', OUTPUT_DIR)

## 4. Ejecutar prepare_nih_multilabel.py

In [ ]:
prepare_cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts' / 'prepare_nih_multilabel.py'),
    '--metadata-csv', str(METADATA_CSV),
    '--images-root', str(IMAGES_ROOT),
    '--labels-json', str(REPO_DIR / 'artifacts' / 'labels.json'),
    '--output-csv', str(OUTPUT_DIR / 'nih_multilabel.csv'),
    '--recursive-image-search',
]

subprocess.run(prepare_cmd, check=True)

## 5. Ejecutar create_patient_splits.py

In [ ]:
splits_cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts' / 'create_patient_splits.py'),
    '--input-csv', str(OUTPUT_DIR / 'nih_multilabel.csv'),
    '--output-dir', str(OUTPUT_DIR / 'splits'),
]

subprocess.run(splits_cmd, check=True)

## 6. Mostrar conteo de filas

In [ ]:
processed = pd.read_csv(OUTPUT_DIR / 'nih_multilabel.csv')
train_df = pd.read_csv(OUTPUT_DIR / 'splits' / 'train.csv')
val_df = pd.read_csv(OUTPUT_DIR / 'splits' / 'val.csv')
test_df = pd.read_csv(OUTPUT_DIR / 'splits' / 'test.csv')

pd.DataFrame({
    'split': ['all', 'train', 'val', 'test'],
    'rows': [len(processed), len(train_df), len(val_df), len(test_df)],
    'patients': [
        processed['Patient ID'].astype(str).nunique(),
        train_df['Patient ID'].astype(str).nunique(),
        val_df['Patient ID'].astype(str).nunique(),
        test_df['Patient ID'].astype(str).nunique(),
    ],
})

## 7. Mostrar conteo por etiqueta

In [ ]:
with open(REPO_DIR / 'artifacts' / 'labels.json', 'r', encoding='utf-8') as file:
    LABELS = json.load(file)

processed[LABELS].sum().astype(int).sort_values(ascending=False)

## 8. Verificar que No Finding no es columna

In [ ]:
assert 'No Finding' not in processed.columns
print('OK: No Finding no es columna de etiqueta.')

## 9. Verificar que no hay Patient ID repetido entre train, val y test

In [ ]:
train_ids = set(train_df['Patient ID'].astype(str))
val_ids = set(val_df['Patient ID'].astype(str))
test_ids = set(test_df['Patient ID'].astype(str))

assert not (train_ids & val_ids)
assert not (train_ids & test_ids)
assert not (val_ids & test_ids)

print('OK: no hay fuga de pacientes entre splits.')

## 10. Guardar resultados en /kaggle/working/processed

Los archivos esperados quedan en:

- `/kaggle/working/processed/nih_multilabel.csv`
- `/kaggle/working/processed/splits/train.csv`
- `/kaggle/working/processed/splits/val.csv`
- `/kaggle/working/processed/splits/test.csv`